# 09 — Feature Engineering: anti-leakage pressure features for shot prediction

Builds the feature table `gold.match_momentum_features` from the raw `gold.match_momentum` table. Core design principle: all pressure features use **only observations from minutes ≤ t−1**; the target variable (shot/goal at minute t) comes from minute t. The current-minute `pressure(t)` is deliberately excluded from the feature table.

Sections:
1. Opponent pressure (self-join on same minute, different team)
2. Time windows (rowsBetween: −3/−5/−10 → −1)
3. Pressure lags (t−1 to t−5)
4. Rolling statistics (mean, max, sd) — no NULL imputation
5. Pressure change and advantage
6. Match phase (minute binning)
7. Target variables — with explicit coalesce(NULL → 0)
8. Final feature table (excluding current-minute pressure)
9. Target quality checks
10. target_shot distribution
11. Filter for first model (complete 5-minute window)
12. Descriptive comparison target=0 vs target=1


### 0. Configuration 

Reads `gold.match_momentum`. `FEATURE_TABLE` is the target Gold feature table.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

MOMENTUM_TABLE = "wsl_analytics.gold.match_momentum"
FEATURE_TABLE = "wsl_analytics.gold.match_momentum_features"

momentum_df = spark.table(MOMENTUM_TABLE)

print("match_momentum number of records:", momentum_df.count())
display(momentum_df)


### Schema 

Verifies the source table schema.

In [ ]:
momentum_df.printSchema()


## 1. Opponent pressure

For every fixture × team × minute, include the opposing team's pressure for that exact minute.


In [ ]:
opponent_pressure_df = (
    momentum_df
    .select(
        "sportmonks_fixture_id",
        "sportmonks_team_id",
        "minute",
        F.col("pressure").alias("opponent_pressure")
    )
)

features_base_df = (
    momentum_df.alias("own")
    .join(
        opponent_pressure_df.alias("opp"),
        (
            (F.col("own.sportmonks_fixture_id") == F.col("opp.sportmonks_fixture_id"))
            &
            (F.col("own.minute") == F.col("opp.minute"))
            &
            (F.col("own.sportmonks_team_id") != F.col("opp.sportmonks_team_id"))
        ),
        how="left"
    )
    .select(
        F.col("own.*"),
        F.col("opp.opponent_pressure")
    )
)

display(features_base_df)


## 2. Time windows

`rowsBetween(-5, -1)` means the five preceding rows and excludes the current minute. Since match_momentum has a complete minute-by-minute grid, this corresponds to the five preceding minutes.


In [ ]:
team_window = (
    Window
    .partitionBy(
        "sportmonks_fixture_id",
        "sportmonks_team_id"
    )
    .orderBy("minute")
)

window_prev_3 = team_window.rowsBetween(-3, -1)
window_prev_5 = team_window.rowsBetween(-5, -1)
window_prev_10 = team_window.rowsBetween(-10, -1)


## 3. Pressure lags


In [ ]:
features_df = (
    features_base_df

    .withColumn("pressure_lag_1", F.lag("pressure", 1).over(team_window))
    .withColumn("pressure_lag_2", F.lag("pressure", 2).over(team_window))
    .withColumn("pressure_lag_3", F.lag("pressure", 3).over(team_window))
    .withColumn("pressure_lag_4", F.lag("pressure", 4).over(team_window))
    .withColumn("pressure_lag_5", F.lag("pressure", 5).over(team_window))

    .withColumn(
        "opponent_pressure_lag_1",
        F.lag("opponent_pressure", 1).over(team_window)
    )
    .withColumn(
        "opponent_pressure_lag_3",
        F.lag("opponent_pressure", 3).over(team_window)
    )
    .withColumn(
        "opponent_pressure_lag_5",
        F.lag("opponent_pressure", 5).over(team_window)
    )
)


## 4. Rolling features

Missing `pressure` values are not imputed with zero. The additional `*_n` columns indicate the number of actually available observations that formed the given window.

In [ ]:
features_df = (
    features_df

    # średnie
    .withColumn("pressure_prev_3m_mean", F.avg("pressure").over(window_prev_3))
    .withColumn("pressure_prev_5m_mean", F.avg("pressure").over(window_prev_5))
    .withColumn("pressure_prev_10m_mean", F.avg("pressure").over(window_prev_10))

    # maksima
    .withColumn("pressure_prev_3m_max", F.max("pressure").over(window_prev_3))
    .withColumn("pressure_prev_5m_max", F.max("pressure").over(window_prev_5))
    .withColumn("pressure_prev_10m_max", F.max("pressure").over(window_prev_10))

    # zmienność
    .withColumn("pressure_prev_5m_sd", F.stddev_samp("pressure").over(window_prev_5))
    .withColumn("pressure_prev_10m_sd", F.stddev_samp("pressure").over(window_prev_10))

    # liczba dostępnych obserwacji
    .withColumn("pressure_prev_3m_n", F.count("pressure").over(window_prev_3))
    .withColumn("pressure_prev_5m_n", F.count("pressure").over(window_prev_5))
    .withColumn("pressure_prev_10m_n", F.count("pressure").over(window_prev_10))

    # pressure przeciwnika
    .withColumn(
        "opponent_pressure_prev_5m_mean",
        F.avg("opponent_pressure").over(window_prev_5)
    )
    .withColumn(
        "opponent_pressure_prev_5m_n",
        F.count("opponent_pressure").over(window_prev_5)
    )
)


## 5. Pressure change and advantage


In [ ]:
features_df = (
    features_df

    .withColumn(
        "pressure_change_3m",
        F.col("pressure_lag_1") - F.col("pressure_lag_3")
    )
    .withColumn(
        "pressure_change_5m",
        F.col("pressure_lag_1") - F.col("pressure_lag_5")
    )

    .withColumn(
        "pressure_advantage_lag_1",
        F.col("pressure_lag_1") - F.col("opponent_pressure_lag_1")
    )
    .withColumn(
        "pressure_advantage_prev_5m",
        F.col("pressure_prev_5m_mean") - F.col("opponent_pressure_prev_5m_mean")
    )
)


In [ ]:
[col for col in features_df.columns if "pressure" in col]


## 6. Match phase (minute binning)


In [ ]:
features_df = (
    features_df

    .withColumn(
        "match_phase",
        F.when(F.col("minute") <= 15, "0_15")
         .when(F.col("minute") <= 30, "16_30")
         .when(F.col("minute") <= 45, "31_45")
         .when(F.col("minute") <= 60, "46_60")
         .when(F.col("minute") <= 75, "61_75")
         .otherwise("76_plus")
    )

    .withColumn(
        "second_half",
        (F.col("minute") > 45).cast("int")
    )
)


## 7. Target variables — with explicit coalesce(NULL → 0)

In match_momentum, minutes with no events can contain NULL in the shot table columns. Thus, we explicitly map missing shots to 0 using coalesce.Consequently, target_shot has only 0/1 values rather than NULL/1.


In [ ]:
features_df = (
    features_df

    .withColumn(
        "target_shot",
        F.when(
            F.coalesce(F.col("shot_count"), F.lit(0)) > 0,
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "target_shot_count",
        F.coalesce(
            F.col("shot_count"),
            F.lit(0)
        ).cast("long")
    )

    .withColumn(
        "target_xg",
        F.coalesce(
            F.col("shot_xg"),
            F.lit(0.0)
        ).cast("double")
    )

    .withColumn(
        "target_shot_on_target",
        F.when(
            F.coalesce(F.col("shots_on_target"), F.lit(0)) > 0,
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "target_goal",
        F.when(
            F.coalesce(F.col("goal_count"), F.lit(0)) > 0,
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)


## 8. Final feature table (excluding current-minute pressure)

Current `pressure` is excluded from `features_final_df` to prevent it from accidentally entering the model as a predictor for minute `t`.


In [ ]:
features_final_df = (
    features_df
    .select(
        # identifiers
        "sportmonks_fixture_id",
        "fotmob_match_id",
        "match_date",
        "sportmonks_team_id",
        "fotmob_team_id",
        "sportmonks_team_name",
        "fotmob_team_name",
        "location",
        "minute",
        "match_phase",
        "second_half",

        # pressure lags
        "pressure_lag_1",
        "pressure_lag_2",
        "pressure_lag_3",
        "pressure_lag_4",
        "pressure_lag_5",

        # rolling pressure
        "pressure_prev_3m_mean",
        "pressure_prev_5m_mean",
        "pressure_prev_10m_mean",
        "pressure_prev_3m_max",
        "pressure_prev_5m_max",
        "pressure_prev_10m_max",
        "pressure_prev_5m_sd",
        "pressure_prev_10m_sd",

        # changes
        "pressure_change_3m",
        "pressure_change_5m",

        # opponent / relative pressure
        "opponent_pressure_lag_1",
        "opponent_pressure_lag_3",
        "opponent_pressure_lag_5",
        "opponent_pressure_prev_5m_mean",
        "pressure_advantage_lag_1",
        "pressure_advantage_prev_5m",

        # data availability
        "pressure_prev_3m_n",
        "pressure_prev_5m_n",
        "pressure_prev_10m_n",
        "opponent_pressure_prev_5m_n",

        # targets
        "target_shot",
        "target_shot_count",
        "target_xg",
        "target_shot_on_target",
        "target_goal"
    )
)

display(features_final_df)


## 9. Target quality checks

All values below should be equal to `0`.


In [ ]:
target_columns = [
    "target_shot",
    "target_shot_count",
    "target_xg",
    "target_shot_on_target",
    "target_goal"
]

target_nulls_df = features_final_df.select(
    *[
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in target_columns
    ]
)

display(target_nulls_df)


## 10. `target_shot` distribution

The percentage is calculated relative to the previously computed record count.


In [ ]:
total_rows = features_final_df.count()

target_distribution_df = (
    features_final_df
    .groupBy("target_shot")
    .count()
    .withColumn(
        "pct",
        F.round(
            F.col("count") / F.lit(total_rows) * 100,
            2
        )
    )
    .orderBy("target_shot")
)

display(target_distribution_df)


## 11. Filter for first model (complete 5-minute window)

This filter keeps records with a complete 5-minute history of their own pressure. If the model also uses the opponent's average pressure, opponent_pressure_prev_5m_n == 5 can be additionally required.


In [ ]:
modeling_df = (
    features_final_df
    .filter(
        F.col("pressure_prev_5m_n") == 5
    )
)

print("Wszystkie rekordy:", features_final_df.count())
print("Pełne 5 minut historii own pressure:", modeling_df.count())


## 12. Descriptive comparison target=0 vs target=1


In [ ]:
display(
    features_final_df
    .groupBy("target_shot")
    .agg(
        F.count("*").alias("n"),
        F.avg("pressure_prev_5m_mean").alias("mean_pressure_prev_5m"),
        F.avg("pressure_prev_5m_max").alias("mean_max_pressure_prev_5m"),
        F.avg("pressure_change_5m").alias("mean_pressure_change_5m"),
        F.avg("pressure_advantage_prev_5m").alias("mean_pressure_advantage_prev_5m")
    )
    .orderBy("target_shot")
)


In [ ]:
display(features_final_df)

## 13. Write a gold table


In [ ]:
(
    features_final_df
    .write
    .mode("overwrite")
    .saveAsTable(FEATURE_TABLE)
)

print(f"Zapisano: {FEATURE_TABLE}")
print("Liczba rekordów:", spark.table(FEATURE_TABLE).count())
